# Data Preprocessing and Feature Engineering

## Purpose

This notebook prepares the cleaned Phase I diabetes dataset for the machine-learning experiments in Phase II. It begins by validating the input data and documenting the predictor types before any transformations are applied.

The later preprocessing steps will address the representation of binary, ordinal, continuous, and count-based variables, as well as the imbalance in the `Diabetes_binary` target. Any transformation or class-imbalance technique that learns information from the data will be applied only to the training set to prevent data leakage.


## 1. Load the Cleaned Dataset

The cleaned dataset produced during Phase I is loaded as the input for Phase II preprocessing. Initial checks are performed before any new transformations are applied.

In [2]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/processed/diabetes_cleaned.csv")

df = pd.read_csv(data_path)

print(f"Dataset loaded from: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.head()

Dataset loaded from: ../data/processed/diabetes_cleaned.csv
Rows: 253,680
Columns: 22


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## 2. Initial Data Validation

The structure and quality of the cleaned dataset are checked again before preprocessing. This confirms that the Phase I output has been loaded correctly and that no unexpected missing values or structural changes have been introduced.

In [3]:
validation_summary = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Values": df.isna().sum(),
    "Unique_Values": df.nunique()})

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Total missing values: {df.isna().sum().sum():,}")
print(f"Exact matching rows: {df.duplicated().sum():,}")

display(validation_summary)

Number of rows: 253,680
Number of columns: 22
Total missing values: 0
Exact matching rows: 24,206


,Data_Type,Missing_Values,Unique_Values
HighBP,int64,0,2
HighChol,int64,0,2
CholCheck,int64,0,2
BMI,int64,0,84
Smoker,int64,0,2
Stroke,int64,0,2
HeartDiseaseorAttack,int64,0,2
PhysActivity,int64,0,2
Fruits,int64,0,2
Veggies,int64,0,2


## 3. Target Distribution

The distribution of `Diabetes_binary` is checked because a large difference between the two classes can affect model training and make ordinary accuracy misleading.

In [4]:
target_summary = (
    df["Diabetes_binary"]
    .value_counts()
    .sort_index()
    .rename_axis("Diabetes_binary")
    .reset_index(name="Respondents"))

target_summary["Class"] = target_summary["Diabetes_binary"].map({
    0: "No diabetes",
    1: "Prediabetes/Diabetes"})

target_summary["Percentage"] = (target_summary["Respondents"] / len(df) * 100).round(2)

target_summary = target_summary[["Diabetes_binary", "Class", "Respondents", "Percentage"]]

display(target_summary.style.hide(axis="index"))

Diabetes_binary,Class,Respondents,Percentage
0,No diabetes,218334,86.070000
1,Prediabetes/Diabetes,35346,13.930000


### Interpretation

The cleaned dataset retains the class imbalance identified during Phase I. Most respondents belong to the no-diabetes class (`86.07%`), while the combined prediabetes/diabetes class represents only `13.93%` of the dataset.

This imbalance will be considered during model training and evaluation. Any resampling or class-balancing method will be applied only to the training data so that the test data continues to represent the original population distribution.

## 4. Predictor Types

The predictors are grouped according to how their values should be interpreted. Although every column is stored numerically in the CSV file, the numbers do not all represent the same type of information. Identifying these groups helps prevent inappropriate transformations during preprocessing.

In [5]:
binary_features = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex"]

ordinal_features = ["GenHlth", "Age", "Education", "Income"]

continuous_features = ["BMI"]

count_features = ["MentHlth", "PhysHlth"]

target = "Diabetes_binary"

feature_groups = pd.DataFrame({
    "Feature_Group": [
        "Binary", "Ordinal", "Continuous", "Count-based", "Target"
    ],
    "Features": [
        ", ".join(binary_features),
        ", ".join(ordinal_features),
        ", ".join(continuous_features),
        ", ".join(count_features),
        target
    ],
    "Number_of_Features": [
        len(binary_features),
        len(ordinal_features),
        len(continuous_features),
        len(count_features),
        1
    ]
})

classified_columns = (
    binary_features
    + ordinal_features
    + continuous_features
    + count_features
    + [target])

print(f"Columns classified: {len(classified_columns)}")
print(f"Dataset columns: {df.shape[1]}")
print(f"All columns accounted for: {set(classified_columns) == set(df.columns)}")

display(feature_groups.style.hide(axis="index"))

Columns classified: 22
Dataset columns: 22
All columns accounted for: True


Feature_Group,Features,Number_of_Features
Binary,"HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex",14
Ordinal,"GenHlth, Age, Education, Income",4
Continuous,BMI,1
Count-based,"MentHlth, PhysHlth",2
Target,Diabetes_binary,1


## 5. Validity of Category Codes and Value Ranges

Each binary and ordinal variable is checked against its documented set of permitted values. The physical health and mental health variables are also checked because they represent number of days and should range from 0 to 30.

In [6]:
expected_values = {
    **{feature: {0, 1} for feature in binary_features},
    "GenHlth": {1, 2, 3, 4, 5},
    "Age": set(range(1, 14)),
    "Education": set(range(1, 7)),
    "Income": set(range(1, 9)),
    "MentHlth": set(range(0, 31)),
    "PhysHlth": set(range(0, 31)),
    "Diabetes_binary": {0, 1}}

validity_results = []

for column, permitted in expected_values.items():
    observed = set(df[column].dropna().unique())
    invalid = observed - permitted

    validity_results.append({
        "Variable": column,
        "Observed_Min": df[column].min(),
        "Observed_Max": df[column].max(),
        "Invalid_Values": sorted(invalid) if invalid else "None",
        "Valid": len(invalid) == 0})

validity_summary = pd.DataFrame(validity_results)

display(validity_summary.style.hide(axis="index"))

print(
    "All coded variables contain permitted values:",
    validity_summary["Valid"].all())

Variable,Observed_Min,Observed_Max,Invalid_Values,Valid
HighBP,0,1,None,True
HighChol,0,1,None,True
CholCheck,0,1,None,True
Smoker,0,1,None,True
Stroke,0,1,None,True
HeartDiseaseorAttack,0,1,None,True
PhysActivity,0,1,None,True
Fruits,0,1,None,True
Veggies,0,1,None,True
HvyAlcoholConsump,0,1,None,True


All coded variables contain permitted values: True


## 6. Investigation of Potential Outliers

BMI is the only continuous variable in the dataset. Its values are reviewed to identify unusually low or high observations. These unusual values are investigated as possible outliers, but they are not automatically treated as errors or removed.

In [7]:
bmi_summary = df["BMI"].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).to_frame(name="BMI")

q1 = df["BMI"].quantile(0.25)
q3 = df["BMI"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

potential_bmi_outliers = (
    (df["BMI"] < lower_bound)
    | (df["BMI"] > upper_bound)).sum()

display(bmi_summary)

print(f"IQR lower boundary: {lower_bound:.2f}")
print(f"IQR upper boundary: {upper_bound:.2f}")
print(f"Potential BMI outliers: {potential_bmi_outliers:,}")
print(f"Observed BMI range: {df['BMI'].min()} to {df['BMI'].max()}")

,BMI
count,253680.000000
mean,28.382364
std,6.608694
min,12.000000
1%,18.000000
25%,24.000000
50%,27.000000
75%,31.000000
99%,50.000000
max,98.000000


IQR lower boundary: 13.50
IQR upper boundary: 41.50
Potential BMI outliers: 9,847
Observed BMI range: 12 to 98


### BMI Outlier Decision

The IQR method identified 9,847 BMI observations outside the range of 13.5 to 41.5. However, this method only identifies values that are unusual compared with most of the dataset, it does not prove that they are incorrect.

The observed BMI values range from 12 to 98 and were already retained during Phase I because there was no evidence that they resulted from data-entry errors. They will remain in the dataset because unusually high or low BMI values may contain useful information for diabetes risk prediction. Their treatment through scaling or discretisation will be considered after the shared training and test data have been established.

## 7. Duplicate-Row Review

The dataset contains rows with identical values across all 22 columns. Because the survey does not provide a unique participant identifier, identical rows cannot be confirmed as accidental duplicate records. Different respondents may have provided the same answers, particularly because most variables contain only a small number of possible values.

In [8]:
matching_rows = df.duplicated().sum()
matching_percentage = matching_rows / len(df) * 100

print(f"Exact matching rows: {matching_rows:,}")
print(f"Percentage of dataset: {matching_percentage:.2f}%")
print(f"Rows remaining if removed: {len(df) - matching_rows:,}")

Exact matching rows: 24,206
Percentage of dataset: 9.54%
Rows remaining if removed: 229,474


### Duplicate-Row Decision

The 24,206 matching rows represent approximately 9.54% of the dataset. These rows will be retained because there is no participant identifier that can confirm they are duplicate submissions. Removing them could incorrectly discard valid respondents and change the original class distribution.

## 8. Proposed Preprocessing Plan

The validation results show that the dataset does not require missing value imputation, duplicate removal, or correction of invalid category codes. The remaining preprocessing decisions therefore focus on representing the different feature types appropriately and preparing the data for fair model training.

Transformations that learn information from the dataset will be fitted using the training data only. The same fitted transformations will then be applied to the test data.

In [9]:
preprocessing_plan = pd.DataFrame({
    "Feature_Group": [
        "Binary",
        "Ordinal",
        "Continuous",
        "Count-based",
        "Target imbalance"
    ],
    "Variables": [
        ", ".join(binary_features),
        ", ".join(ordinal_features),
        ", ".join(continuous_features),
        ", ".join(count_features),
        "Diabetes_binary"
    ],
    "Proposed_Handling": [
        "Retain existing 0/1 encoding",
        "Retain ordered codes and preserve their ranking",
        "Retain BMI; assess scaling or discretisation after the data split",
        "Retain 0–30 values; assess scaling or discretisation after the data split",
        "Apply any balancing method to the training data only"
    ],
    "Reason": [
        "Values already represent two valid categories",
        "The numbers represent ordered categories rather than exact measurements",
        "BMI is valid but has a wider numerical range than most predictors",
        "These variables represent numbers of unhealthy days",
        "The test data must retain the original class distribution"
    ]
})

display(preprocessing_plan.style.hide(axis="index"))

Feature_Group,Variables,Proposed_Handling,Reason
Binary,"HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex",Retain existing 0/1 encoding,Values already represent two valid categories
Ordinal,"GenHlth, Age, Education, Income",Retain ordered codes and preserve their ranking,The numbers represent ordered categories rather than exact measurements
Continuous,BMI,Retain BMI; assess scaling or discretisation after the data split,BMI is valid but has a wider numerical range than most predictors
Count-based,"MentHlth, PhysHlth",Retain 0–30 values; assess scaling or discretisation after the data split,These variables represent numbers of unhealthy days
Target imbalance,Diabetes_binary,Apply any balancing method to the training data only,The test data must retain the original class distribution


## 9. Shared Training and Test Split

The same training and test split used for the original eLCS baseline is reproduced below. Using the same split ensures that the baseline and improved systems are evaluated on exactly the same test records.

The dataset is divided into 80% training data and 20% test data using a fixed random seed of 42. Stratification preserves the original class proportions in both sets. The test data will remain unchanged, while any balancing or learned preprocessing will be applied only to the training data.

In [10]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining class percentages:")
print(
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2))

print("\nTesting class percentages:")
print(
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2))

Training features: (202944, 21)
Testing features: (50736, 21)
Training target: (202944,)
Testing target: (50736,)

Training class percentages:
Diabetes_binary
0    86.07
1    13.93
Name: proportion, dtype: float64

Testing class percentages:
Diabetes_binary
0    86.07
1    13.93
Name: proportion, dtype: float64


### Split Verification

The stratified split preserved the original class distribution in both datasets. The training set contains 202944 records, while the untouched test set contains 50736 records. This same split configuration is used by the original eLCS baseline, allowing later systems to be compared fairly on equivalent test data.

## 10. Training-Class Balancing

The original eLCS baseline predicted almost every record as belonging to the majority no diabetes class. To give the improved system an equal opportunity to learn rules for both outcomes, random undersampling is applied to the training data.

Only the majority class in the training set is reduced. The test set remains unchanged so that evaluation continues to represent the original population distribution.

In [12]:
# Combine the training predictors and target temporarily
training_data = X_train.copy()
training_data["Diabetes_binary"] = y_train

majority_class = training_data[training_data["Diabetes_binary"] == 0]

minority_class = training_data[training_data["Diabetes_binary"] == 1]

# Randomly reduce the majority class to the minority-class size
majority_undersampled = majority_class.sample(
    n=len(minority_class),
    random_state=42)

balanced_training_data = pd.concat(
    [majority_undersampled, minority_class]
).sample(
    frac=1,
    random_state=42
)

X_train_balanced = balanced_training_data.drop(columns=["Diabetes_binary"])

y_train_balanced = balanced_training_data["Diabetes_binary"]

print("Original training rows:", len(X_train))
print("Balanced training rows:", len(X_train_balanced))

print("\nOriginal training class counts:")
print(y_train.value_counts().sort_index())

print("\nBalanced training class counts:")
print(y_train_balanced.value_counts().sort_index())

print("\nTest rows remain unchanged:", len(X_test))

Original training rows: 202944
Balanced training rows: 56554

Original training class counts:
Diabetes_binary
0    174667
1     28277
Name: count, dtype: int64

Balanced training class counts:
Diabetes_binary
0    28277
1    28277
Name: count, dtype: int64

Test rows remain unchanged: 50736


### Balancing Result

Random undersampling reduced the training set from 202944 to 56554 records, with 28277 records from each class. This gives the improved eLCS system equal exposure to the no diabetes and prediabetes/diabetes classes during training.

No records were removed from the test set. It remains at 50736 records with the original class distribution, allowing the improved system to be evaluated under realistic conditions and compared fairly with the original baseline.

## 11. Final Feature Representation

All predictors are already stored numerically and contain valid values. The binary variables retain their existing 0/1 encoding, while the ordinal variables retain their ordered category codes.

BMI, mental-health days, and physical-health days remain in their original numerical form because eLCS can construct range based rules for numerical attributes. Scaling is not required because eLCS does not calculate distance between observations in the same way as methods such as k-nearest neighbours.

No predictors are removed at this stage. Retaining all 21 predictors allows the improved eLCS system to determine which attributes are useful when forming classification rules. The principal preprocessing improvement is therefore the balancing of the training classes, while the test data and feature meanings remain unchanged.

In [13]:
print("Balanced training features:", X_train_balanced.shape)
print("Test features:", X_test.shape)

print(
    "Training and test columns match:",
    list(X_train_balanced.columns) == list(X_test.columns))

print(
    "Missing values in balanced training data:",
    X_train_balanced.isna().sum().sum())

print(
    "Missing values in test data:",
    X_test.isna().sum().sum())

print(
    "All training predictors are numeric:",
    X_train_balanced.dtypes.apply(
        lambda dtype: pd.api.types.is_numeric_dtype(dtype)
    ).all())

Balanced training features: (56554, 21)
Test features: (50736, 21)
Training and test columns match: True
Missing values in balanced training data: 0
Missing values in test data: 0
All training predictors are numeric: True


## 12. Export of Prepared Data

The balanced training data and unchanged test data are exported as separate files. This preserves the distinction between data used for model learning and data reserved for final evaluation.

The target variable is included as the final column in both files. These files will be used when developing and evaluating the improved eLCS system.

In [14]:
output_directory = Path("../data/processed/elcs")
output_directory.mkdir(parents=True, exist_ok=True)

prepared_training_data = X_train_balanced.copy()
prepared_training_data["Diabetes_binary"] = y_train_balanced

prepared_test_data = X_test.copy()
prepared_test_data["Diabetes_binary"] = y_test

training_output_path = (output_directory / "diabetes_elcs_training_balanced.csv")

test_output_path = (output_directory / "diabetes_elcs_test_unchanged.csv")

prepared_training_data.to_csv(
    training_output_path,
    index=False)

prepared_test_data.to_csv(
    test_output_path,
    index=False)

print(f"Training data exported to: {training_output_path}")
print(f"Training shape: {prepared_training_data.shape}")

print(f"\nTest data exported to: {test_output_path}")
print(f"Test shape: {prepared_test_data.shape}")

Training data exported to: ../data/processed/elcs/diabetes_elcs_training_balanced.csv
Training shape: (56554, 22)

Test data exported to: ../data/processed/elcs/diabetes_elcs_test_unchanged.csv
Test shape: (50736, 22)


## 13. Expected Effect on eLCS Performance

The original eLCS baseline was trained on a strongly imbalanced dataset and consequently predicted almost every test record as belonging to the no-diabetes class. Balancing the training data gives eLCS equal exposure to both outcomes, which should help it discover more rules describing respondents with prediabetes or diabetes.

This change is expected to improve minority class recall, F1-score, and balanced accuracy. However, reducing the majority class may also lower ordinary accuracy or increase false positive predictions. The improvement must therefore be confirmed by training the improved eLCS system and evaluating it on the unchanged test set.

The original feature meanings and values have been preserved, allowing the resulting eLCS rules to remain understandable. The fixed split, training-only balancing, and unchanged test set also prevent data leakage and support a fair comparison with the original baseline.

## 14. Secondary SMOTENC Dataset

A second balanced training dataset is created using SMOTENC as an additional comparison. Unlike ordinary SMOTE, SMOTENC distinguishes between categorical and continuous predictors. This prevents synthetic records from containing invalid intermediate category values, such as a smoking status between 0 and 1.

All binary, ordinal, and count based predictors are treated as categorical during oversampling, while BMI is treated as continuous. SMOTENC is applied only to the training set. The shared test set remains unchanged.

In [17]:
from imblearn.over_sampling import SMOTENC

categorical_columns = (
    binary_features
    + ordinal_features
    + count_features)

categorical_indices = [
    X_train.columns.get_loc(column)
    for column in categorical_columns]

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42)

X_train_smotenc, y_train_smotenc = smotenc.fit_resample(
    X_train,
    y_train)

print("Original training shape:", X_train.shape)
print("SMOTENC training shape:", X_train_smotenc.shape)

print("\nOriginal training class counts:")
print(y_train.value_counts().sort_index())

print("\nSMOTENC training class counts:")
print(y_train_smotenc.value_counts().sort_index())

print("\nUnchanged test shape:", X_test.shape)

Original training shape: (202944, 21)
SMOTENC training shape: (349334, 21)

Original training class counts:
Diabetes_binary
0    174667
1     28277
Name: count, dtype: int64

SMOTENC training class counts:
Diabetes_binary
0    174667
1    174667
Name: count, dtype: int64

Unchanged test shape: (50736, 21)


In [18]:
smotenc_validity_results = []

for column in categorical_columns:
    original_values = set(X_train[column].unique())
    smotenc_values = set(X_train_smotenc[column].unique())
    invalid_values = smotenc_values - original_values

    smotenc_validity_results.append({
        "Variable": column,
        "Invalid_Values": (
            sorted(invalid_values)
            if invalid_values
            else "None"
        ),
        "Valid": len(invalid_values) == 0
    })

smotenc_validity_summary = pd.DataFrame(smotenc_validity_results)

display(smotenc_validity_summary.style.hide(axis="index"))

print(
    "All categorical variables contain valid values:",
    smotenc_validity_summary["Valid"].all())

print(
    "Missing values in SMOTENC training data:",
    X_train_smotenc.isna().sum().sum())

print(
    "SMOTENC BMI range:",
    round(X_train_smotenc["BMI"].min(), 2),
    "to",
    round(X_train_smotenc["BMI"].max(), 2))

Variable,Invalid_Values,Valid
HighBP,None,True
HighChol,None,True
CholCheck,None,True
Smoker,None,True
Stroke,None,True
HeartDiseaseorAttack,None,True
PhysActivity,None,True
Fruits,None,True
Veggies,None,True
HvyAlcoholConsump,None,True


All categorical variables contain valid values: True
Missing values in SMOTENC training data: 0
SMOTENC BMI range: 12 to 98


### SMOTENC Validation

SMOTENC produced a balanced training set containing 174667 records from each class. The validation checks confirmed that the synthetic records retained permitted values for every binary, ordinal, and count based predictor. No missing values were introduced, while BMI remained within the observed training range.

This dataset will be used only as a secondary experiment. The random undersampled dataset remains the primary preprocessing approach, and both approaches will be evaluated using the same unchanged test set.

In [19]:
prepared_smotenc_training_data = (X_train_smotenc.copy())

prepared_smotenc_training_data[
    "Diabetes_binary"
] = y_train_smotenc.to_numpy()

smotenc_output_path = (
    output_directory
    / "diabetes_elcs_training_smotenc.csv")

prepared_smotenc_training_data.to_csv(
    smotenc_output_path,
    index=False)

print(
    f"SMOTENC training data exported to: "
    f"{smotenc_output_path}")

print(
    "SMOTENC training shape:",
    prepared_smotenc_training_data.shape)

print("\nSMOTENC class counts:")

print(
    prepared_smotenc_training_data[
        "Diabetes_binary"
    ].value_counts().sort_index())

SMOTENC training data exported to: ../data/processed/elcs/diabetes_elcs_training_smotenc.csv
SMOTENC training shape: (349334, 22)

SMOTENC class counts:
Diabetes_binary
0    174667
1    174667
Name: count, dtype: int64
